MLFlowClient nos permite interactuar con experimentos y modelos guardados en el servidor

In [1]:
from mlflow.tracking import MlflowClient

MLFLOW_TRACKLING_URI = 'sqlite:///mlflow.db'

client = MlflowClient(tracking_uri=MLFLOW_TRACKLING_URI)

2025/11/11 10:02:29 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2025/11/11 10:02:29 INFO mlflow.store.db.utils: Updating database tables
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


Obtener los experimentos existentes

In [3]:
client.search_experiments()

[<Experiment: artifact_location='/workspaces/mlops-zoomcamp/02-experiment-tracking/mlruns/2', creation_time=1762437757019, experiment_id='2', last_update_time=1762437757019, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}>,
 <Experiment: artifact_location='mlflow-artifacts:/0', creation_time=1762437075843, experiment_id='0', last_update_time=1762437075843, lifecycle_stage='active', name='Default', tags={}>]

Crear experimentos nuevos

In [4]:
client.create_experiment(name='my-cool-experiment')

'3'

In [5]:
from mlflow.entities import ViewType

Ver las ejecuciones dentro de un experimento concreto, ordenándolas (order_by) y filtrándolas (filter string)

In [10]:
runs = client.search_runs(
    experiment_ids='2',
    filter_string='',
    run_view_type=ViewType.ACTIVE_ONLY,
    max_results=5,
    order_by=['metrics.rmse ASC']
)

In [11]:
for run in runs:
    print(f"run id: {run.info.run_id}, rmse: {run.data.metrics['rmse']}")

run id: fbb4e6d01d0b4fd4aa5549ad6fc20a9f, rmse: 6.318445793399953
run id: 3b7426c7bcb5491ca5e14aa7cd47fa3d, rmse: 6.318445793399953
run id: efee4056a3904239a09796bb77b31534, rmse: 6.3250186081862045
run id: 4e5c3b06c4f54819a4632c041ba0d3fd, rmse: 6.3661914818743925
run id: 96f73665c7fd4a40a26c8ff4e1d3d8e4, rmse: 6.409080835629773


In [12]:
runs = client.search_runs(
    experiment_ids='2',
    filter_string='metrics.rmse < 6.4',
    run_view_type=ViewType.ACTIVE_ONLY,
    max_results=5,
    order_by=['metrics.rmse ASC']
)

In [13]:
for run in runs:
    print(f"run id: {run.info.run_id}, rmse: {run.data.metrics['rmse']}")

run id: fbb4e6d01d0b4fd4aa5549ad6fc20a9f, rmse: 6.318445793399953
run id: 3b7426c7bcb5491ca5e14aa7cd47fa3d, rmse: 6.318445793399953
run id: efee4056a3904239a09796bb77b31534, rmse: 6.3250186081862045
run id: 4e5c3b06c4f54819a4632c041ba0d3fd, rmse: 6.3661914818743925


In [14]:
import mlflow

mlflow.set_tracking_uri(MLFLOW_TRACKLING_URI)

Podemos usar los artifacts de una ejecución de un experimento para registrar un modelo. Si el modelo indicado no existe, lo crea. 

In [19]:
run_id = "fbb4e6d01d0b4fd4aa5549ad6fc20a9f"
model_uri = f"runs:/{run_id}/models_mlflow"
model_name = "nyc-taxi-regressor"
mlflow.register_model(model_uri=model_uri, name=model_name)

Registered model 'nyc-taxi-regressor' already exists. Creating a new version of this model...
2025/11/11 10:16:42 WARNING mlflow.tracking._model_registry.fluent: Run with id fbb4e6d01d0b4fd4aa5549ad6fc20a9f has no artifacts at artifact path 'models_mlflow', registering model based on models:/m-5482c3fc7a4f428ba1c2f65f565386e6 instead
Created version '1' of model 'nyc-taxi-regressor'.


<ModelVersion: aliases=[], creation_timestamp=1762856202136, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1762856202136, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='fbb4e6d01d0b4fd4aa5549ad6fc20a9f', run_link=None, source='models:/m-5482c3fc7a4f428ba1c2f65f565386e6', status='READY', status_message=None, tags={}, user_id=None, version=1>

Solo tengo artifacts de una ejecución así que no tengo más versiones del modelo, pero podemos iterar las últimas versiones de un modelo:

In [24]:
latest_versions = client.get_latest_versions(name=model_name)

for version in latest_versions:
    print(f"version: {version.version}, stage: {version.current_stage}")

version: 1, stage: None


/tmp/ipykernel_23238/232761966.py:1: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  latest_versions = client.get_latest_versions(name=model_name)


Podemos transicionar un modelo entre stages (Develop, Staging y Production), pero las stages están deprecadas

In [26]:
client.transition_model_version_stage(
    name = model_name,
    version='1',
    stage='Staging',
    archive_existing_versions=False
)

/tmp/ipykernel_23238/453442746.py:1: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


<ModelVersion: aliases=[], creation_timestamp=1762856202136, current_stage='Staging', deployment_job_state=None, description=None, last_updated_timestamp=1762856722921, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='fbb4e6d01d0b4fd4aa5549ad6fc20a9f', run_link=None, source='models:/m-5482c3fc7a4f428ba1c2f65f565386e6', status='READY', status_message=None, tags={}, user_id=None, version=1>

El estándar ahora en mlflow son las tags, que también podemos añadir a una versión de un modelo usando el cliente:

In [28]:
client.set_model_version_tag(
    name = model_name,
    version='1',
    key = 'Stage',
    value = 'Staging'
)

In [30]:
from datetime import datetime

date = datetime.today().date()

Podemos también cambiar la descripción de una versión

In [31]:
client.update_model_version(
    name=model_name,
    version=1,
    description=f"Model version was tagged on {date}"
)

<ModelVersion: aliases=[], creation_timestamp=1762856202136, current_stage='Staging', deployment_job_state=None, description='Model version was tagged on 2025-11-11', last_updated_timestamp=1762857715520, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='fbb4e6d01d0b4fd4aa5549ad6fc20a9f', run_link=None, source='models:/m-5482c3fc7a4f428ba1c2f65f565386e6', status='READY', status_message=None, tags={'Stage': 'Staging'}, user_id=None, version=1>

## Comparing versions and selecting the new "Production" model

In [34]:
from sklearn.metrics import root_mean_squared_error
import pandas as pd

def read_dataframe(url):
    df = pd.read_parquet(url)

    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    return df


def preprocess(df, dv):
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    categorical = ['PU_DO']
    numerical = ['trip_distance']
    train_dicts = df[categorical + numerical].to_dict(orient='records')
    return dv.transform(train_dicts)


def test_model(name, stage, X_test, y_test):
    model = mlflow.pyfunc.load_model(f"models:/{name}/{stage}")
    y_pred = model.predict(X_test)
    return {"rmse": root_mean_squared_error(y_test, y_pred)}

In [35]:
df = read_dataframe('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-03.parquet')

In [36]:
client.download_artifacts(run_id=run_id, path='preprocessor', dst_path='.')

'/workspaces/mlops-zoomcamp/02-experiment-tracking/preprocessor'

In [37]:
import pickle

with open("preprocessor/preprocessor.b", "rb") as f_in:
    dv = pickle.load(f_in)

In [38]:
X_test = preprocess(df, dv)

In [40]:
target = "duration"
y_test = df[target].values

In [42]:
%time test_model(name=model_name, stage="Staging", X_test=X_test, y_test=y_test)

CPU times: user 21.5 s, sys: 197 ms, total: 21.6 s
Wall time: 15.7 s


{'rmse': 6.810292042164062}

Aquí deberíamos testear diferentes versiones o stages del modelo, para decidir qué versión debería ser la de producción comparando tiempo de ejecución, RMSE, etc. Como solo tengo una versión del modelo en el registry, no puedo hacer esa comparación.